# Selected DE genes from downsampling

In [ ]:
#load packages
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import os
from matplotlib.pyplot import rc_context
import gseapy

sc.settings.verbosity = 3             
sc.settings.set_figure_params(dpi=100,figsize=(4,4))
sc.set_figure_params(scanpy=True, fontsize=8, dpi_save=180, format='svg', transparent=True)

In [ ]:


my_c = ['#fe864a', '#c9d179', '#95d0fc', '#fbdd7e', '#388004', '#c760ff', '#76cd26', '#ffbacd', '#d6b4fc',
        '#cfaf7b','#8b88f8', '#acc2d9', '#894585', '#82cbb2', '#d99b82', '#fdee73', '#5170d7', '#ffd8b1', 
        '#b7e1a1', '#82a67d', '#b7fffa', '#b790d4']





**Note**: The genes used to make this plots were sleteced using the results from the notebook DE_Postnatal_Normal. We used the csv files containing the differental genes between each condition and then selected the overlaping genes inside each cell type.

**Eg.**: DE genes found between P2/P3 Vs P11 were selected and the DE genes between P2/P3 Vs P60 were also selected. Then We checked the genes that overlaped between both conditions, and selected the top 20 overlaping genes with the highest log fold change value and highest scores.

**Note**: If the overlaping genes dind't made a total of 20 genes we would select all the overlaping genes plus an equal or almost equal number of genes with the highest logfold change and highest scores value for both conditions, until we had 20 genes.

**Eg.**: We only have 5 overlaping genes, so we will selecte these 5 genes and the 7 or 8 genes with the highest log fold change and highest scores for the condition P2/P3 Vs P11 and the same will be done for the condition P2/P3 Vs P60. In this case one condition will have 7 genes and the other one will have 8 in order to make a total of 20 genes.

**FINAL NOTE**: All the genes were selected using the function ***filter_de_genes*** and had a significant defferential expression (*p-value < 0.05*). The overlaping genes were only counted if they had a score higher that the onegiven to the function parameters (deppending on the cell type this value changed).


In [ ]:


def filter_de_genes(conditions=dict, gene_column=str, score=int):
    conditions_initial = [] #list with the df of each condition without filtering 
    all_genes = {} #dict with all genes by condition being overlaping genes or not, this contains duplicated values 
    all_genes_overlap = {} #dict with the comun genes for each condition, the genes have a score higher than the set value for score
    conditions_names = [] #list with the cames of each condition 

    for k, v in conditions.items():
        conditions_names.append(k)
        filter1 = v[0][v[0]['scores'] > score]
        filter2 = v[1][v[1]['scores'] > score]
        genes1 = list(filter1[gene_column])
        genes2 = list(filter2[gene_column])
        overlap_genes = []
        all_condition_genes = list(v[0][gene_column]) + list(v[1][gene_column])
        for gene in genes1:
            if gene in genes2:
                overlap_genes.append(gene)

        for gene in genes2:
            all_condition_genes.append(gene)

        globals()[k] = pd.concat([v[0], v[1]], axis=0, ignore_index=True)
        conditions_initial.append(globals()[k])
        all_genes_overlap[k] = overlap_genes
        all_genes[k] = all_condition_genes

    genes1 = all_genes[list(all_genes.keys())[0]]
    genes2 = all_genes[list(all_genes.keys())[1]]
    genes3 = all_genes[list(all_genes.keys())[2]]

    o_genes1 = all_genes_overlap[list(all_genes_overlap.keys())[0]]
    o_genes2 = all_genes_overlap[list(all_genes_overlap.keys())[1]]
    o_genes3 = all_genes_overlap[list(all_genes_overlap.keys())[2]]

    print(f"all genes1: {len(set(genes1))}\nall genes2: {len(set(genes2))}\nall genes3: {len(set(genes3))}")
    print(f"commun genes1: {len(set(o_genes1))}\ncommun genes2: {len(set(o_genes2))}\ncommun genes3: {len(set(o_genes3))}")


    overlap_genes = {conditions_names[0]: [], conditions_names[1]: [], conditions_names[2]: []}

    #get the genes that overlap between condition to further remove them

    for gene in genes1:
        idxl1 = conditions_initial[0].index[conditions_initial[0][gene_column] == gene].tolist()
        if len(idxl1) > 1:
            idx1 = idxl1[1]
        else: idx1 = idxl1[0]
        if gene in genes2:
            idxl2 = conditions_initial[1].index[conditions_initial[1][gene_column] == gene].tolist()
            if len(idxl2) > 1:
                idx2 = idxl2[1]
            else: idx2 = idxl2[0]
            if int(conditions_initial[0]['scores'].iloc[idx1]) >= int(conditions_initial[1]['scores'].iloc[idx2]) and conditions_initial[0]['logfoldchanges'].iloc[idx1] > conditions_initial[1]['logfoldchanges'].iloc[idx2]:
                overlap_genes[conditions_names[1]].append(gene)
            else: overlap_genes[conditions_names[0]].append(gene)
        if gene in genes3:
            idxl3 = conditions_initial[2].index[conditions_initial[2][gene_column] == gene].tolist()
            if len(idxl3) > 1:
                idx3 = idxl3[1]
            else: idx3 = idxl3[0]
            if int(conditions_initial[0]['scores'].iloc[idx1]) >= int(conditions_initial[2]['scores'].iloc[idx3]) and conditions_initial[0]['logfoldchanges'].iloc[idx1] > conditions_initial[2]['logfoldchanges'].iloc[idx3]:
                overlap_genes[conditions_names[2]].append(gene)
            else: overlap_genes[conditions_names[0]].append(gene)

    for gene in genes2:
        idxl2 = idx2 = conditions_initial[1].index[conditions_initial[1][gene_column] == gene].tolist()
        if len(idxl2) > 1:
            idx2 = idxl2[1]
        else: idx2 = idxl2[0]
        if gene in genes1:
            idxl1 = conditions_initial[0].index[conditions_initial[0][gene_column] == gene].tolist()
            if len(idxl1) > 1:
                idx1 = idxl1[1]
            else: idx1 = idxl1[0]
            if int(conditions_initial[1]['scores'].iloc[idx2]) >= int(conditions_initial[0]['scores'].iloc[idx1]) and conditions_initial[1]['logfoldchanges'].iloc[idx2] > conditions_initial[0]['logfoldchanges'].iloc[idx1]:
                overlap_genes[conditions_names[0]].append(gene)
            else: overlap_genes[conditions_names[1]].append(gene)
        if gene in genes3:
            idxl3 = conditions_initial[2].index[conditions_initial[2][gene_column] == gene].tolist()
            if len(idxl3) > 1:
                idx3 = idxl3[1]
            else: idx3 = idxl3[0]
            if int(conditions_initial[1]['scores'].iloc[idx2]) >= int(conditions_initial[2]['scores'].iloc[idx3]) and conditions_initial[1]['logfoldchanges'].iloc[idx2] > conditions_initial[2]['logfoldchanges'].iloc[idx3]:
                overlap_genes[conditions_names[2]].append(gene)
            else: overlap_genes[conditions_names[1]].append(gene)

    for gene in genes3:
        idxl3 = conditions_initial[2].index[conditions_initial[2][gene_column] == gene].tolist()
        if len(idxl3) > 1:
            idx3 = idxl3[1]
        else: idx3 = idxl3[0]
        if gene in genes1:
            idxl1 = conditions_initial[0].index[conditions_initial[0][gene_column] == gene].tolist()
            if len(idxl1) > 1:
                idx1 = idxl1[1]
            else: idx1 = idxl1[0]
            if (int(conditions_initial[2]['scores'].iloc[idx3]) >= int(conditions_initial[0]['scores'].iloc[idx1])) and (conditions_initial[2]['logfoldchanges'].iloc[idx3] > conditions_initial[0]['logfoldchanges'].iloc[idx1]):
                overlap_genes[conditions_names[0]].append(gene)
            else: overlap_genes[conditions_names[2]].append(gene)
        if gene in genes2:
            idxl2 = idx2 = conditions_initial[1].index[conditions_initial[1][gene_column] == gene].tolist()
            if len(idxl2) > 1:
                idx2 = idxl2[1]
            else: idx2 = idxl2[0]
            print(gene, int(conditions_initial[0]['scores'].iloc[idx2]), int(conditions_initial[2]['scores'].iloc[idx3]), conditions_initial[1]['logfoldchanges'].iloc[idx2], conditions_initial[2]['logfoldchanges'].iloc[idx3])

            if int(conditions_initial[2]['scores'].iloc[idx3]) >= int(conditions_initial[1]['scores'].iloc[idx2]) and conditions_initial[2]['logfoldchanges'].iloc[idx3] > conditions_initial[1]['logfoldchanges'].iloc[idx2]:
                overlap_genes[conditions_names[1]].append(gene)
            else: overlap_genes[conditions_names[2]].append(gene)

    
    print(f"overlap {conditions_names[0]}: {len(set(overlap_genes[conditions_names[0]]))}\noverlap {conditions_names[1]}: {len(set(overlap_genes[conditions_names[1]]))}\noverlap {conditions_names[2]}: {len(set(overlap_genes[conditions_names[2]]))}")

    # filter out the overlap genes between condition, so we can have unique genes for each condition
    conditions_initial[0] = conditions_initial[0][~conditions_initial[0][gene_column].isin(overlap_genes[conditions_names[0]])]
    conditions_initial[1] = conditions_initial[1][~conditions_initial[1][gene_column].isin(overlap_genes[conditions_names[1]])]
    conditions_initial[2] = conditions_initial[2][~conditions_initial[2][gene_column].isin(overlap_genes[conditions_names[2]])]

    # Get the overlaping genes between the two dataframes for each condition 
    commun_genes1 = all_genes_overlap[list(all_genes_overlap.keys())[0]]
    commun_genes2 = all_genes_overlap[list(all_genes_overlap.keys())[1]]
    commun_genes3 = all_genes_overlap[list(all_genes_overlap.keys())[2]]

    # Filter out all the genes that don't overlap inside the condition
    conditions_initial[0] = conditions_initial[0][conditions_initial[0][gene_column].isin(commun_genes1)]
    conditions_initial[1] = conditions_initial[1][conditions_initial[1][gene_column].isin(commun_genes2)]
    conditions_initial[2] = conditions_initial[2][conditions_initial[2][gene_column].isin(commun_genes3)]

    selected_genes_final = []

    for i in range(len(conditions_names)):
        print(conditions_names[i])
        if len(conditions_initial[i]['names'].unique()) > 20:
            conditions_initial[i].sort_values(by=['scores', 'logfoldchanges'], inplace=True, ascending=False)
            genes_final = list(conditions_initial[i]['names'].unique()[0:20])
            print(len(genes_final))
            selected_genes_final += genes_final
        else:

            selected_commun_genes1 = list(conditions_initial[i][gene_column].unique())
            print(len(selected_commun_genes1))
            print(conditions_names[i])
            raw_genes1 = conditions[conditions_names[i]][0]
            raw_genes2 = conditions[conditions_names[i]][1]
            raw_genes1 = raw_genes1[raw_genes1['scores'] > score]
            raw_genes2 = raw_genes2[raw_genes2['scores'] > score]
            raw_genes1 = raw_genes1[~raw_genes1[gene_column].isin(overlap_genes[conditions_names[i]])]
            raw_genes2 = raw_genes2[~raw_genes2[gene_column].isin(overlap_genes[conditions_names[i]])]
            raw_genes1.sort_values(by=['scores', 'logfoldchanges'], inplace=True, ascending=False)
            raw_genes2.sort_values(by=['scores', 'logfoldchanges'], inplace=True, ascending=False)
            selected_genes1 = [gene for gene in raw_genes1[gene_column] if gene not in selected_commun_genes1]
            selected_genes2= [gene for gene in raw_genes2[gene_column] if gene not in selected_commun_genes1]
            j = 0
            iter = 0
            raw_1 = []
            raw_2 = []
            print(len(selected_genes1), len(selected_genes2))
            while len(conditions_initial[i]['names'].unique()) + j < 20:
                if len(conditions_initial[i]['names'].unique()) + j < 20:
                    raw_1.append(selected_genes1[iter])
                    j += 1
                if len(conditions_initial[i]['names'].unique()) + j < 20:
                    raw_2.append(selected_genes2[iter])
                    j += 1
                    iter += 1
                else: break
            genes_final = raw_2 + selected_commun_genes1 + raw_1
            print(selected_commun_genes1)
            selected_genes_final += genes_final

    return selected_genes_final

# Epithelial cells

In [ ]:
Epithelial = sc.read_h5ad('Epithelial_cells_DE.h5ad')

In [ ]:
P2_P3vsP11 = pd.read_csv('Epithelial_P2_P3vsP11_markers.csv')
P2_P3vsP60 = pd.read_csv('Epithelial_P2_P3vsP60_markers.csv')

P11vsP2_P3 = pd.read_csv('Epithelial_P11vsP2_P3_markers.csv')
P11vsP60 = pd.read_csv('Epithelial_P11vsP60_markers.csv')

P60vsP2_P3 = pd.read_csv('Epithelial_P60vsP2_P3_markers.csv')
P60vsP11 = pd.read_csv('Epithelial_P60vsP11_markers.csv')

In [ ]:
P2_P3vsP11.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)
P2_P3vsP60.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)

P11vsP2_P3.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)
P11vsP60.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)

P60vsP2_P3.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)
P60vsP11.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)

In [ ]:

conditions = {'P2_P3': [P2_P3vsP11, P2_P3vsP60], 'P11': [P11vsP2_P3, P11vsP60], 'P60':[P60vsP2_P3, P60vsP11]}

In [ ]:
genes = filter_de_genes(conditions, 'names', 18)

In [ ]:
sc.pl.heatmap(Epithelial,var_names=genes,groupby="Age", show_gene_labels=True,use_raw=False,
              layer='scaled',dendrogram=False , swap_axes=True, vmax=-4, vmin=4, cmap='BrBG_r',figsize=(4,15), 
              save='_Epithelial_downsampling.svg')

# Endothelial cells 

In [ ]:
Endothelial = sc.read_h5ad('Endothelial_cells_all_DE.h5ad')

In [ ]:
P2_P3vsP11 = pd.read_csv('Endothelial_cells_all_P2_P3vsP11_markers.csv')
P2_P3vsP60 = pd.read_csv('Endothelial_cells_all_P2_P3vsP60_markers.csv')

P11vsP2_P3 = pd.read_csv('Endothelial_cells_all_P11vsP2_P3_markers.csv')
P11vsP60 = pd.read_csv('Endothelial_cells_all_P11vsP60_markers.csv')

P60vsP2_P3 = pd.read_csv('Endothelial_cells_all_P60vsP2_P3_markers.csv')
P60vsP11 = pd.read_csv('Endothelial_cells_all_P60vsP11_markers.csv')

In [ ]:
P2_P3vsP11.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)
P2_P3vsP60.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)

P11vsP2_P3.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)
P11vsP60.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)

P60vsP2_P3.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)
P60vsP11.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)

In [ ]:
conditions = {'P2_P3': [P2_P3vsP11, P2_P3vsP60], 'P11': [P11vsP2_P3, P11vsP60], 'P60':[P60vsP2_P3, P60vsP11]}

In [ ]:
genes = filter_de_genes(conditions, 'names', 5)

In [ ]:
sc.pl.heatmap(Endothelial,var_names=genes,groupby="Age", show_gene_labels=True,use_raw=False,
              layer='scaled',dendrogram=False , swap_axes=True, vmax=-4, vmin=4, cmap='BrBG_r',figsize=(4,15), 
              save='_Endothelial_downsampling.svg')

# Endothelial cells no proliferantion

In [ ]:
Endothelial_no_prolif = sc.read_h5ad('Endothelial_cells_no_prolif_DE.h5ad')

In [ ]:
P2_P3vsP11 = pd.read_csv('Endothelial_cells_no_prolif_P2_P3vsP11_markers.csv')
P2_P3vsP60 = pd.read_csv('Endothelial_cells_no_prolif_P2_P3vsP60_markers.csv')

P11vsP2_P3 = pd.read_csv('Endothelial_cells_no_prolif_P11vsP2_P3_markers.csv')
P11vsP60 = pd.read_csv('Endothelial_cells_no_prolif_P11vsP60_markers.csv')

P60vsP2_P3 = pd.read_csv('Endothelial_cells_no_prolif_P60vsP2_P3_markers.csv')
P60vsP11 = pd.read_csv('Endothelial_cells_no_prolif_P60vsP11_markers.csv')

In [ ]:
P2_P3vsP11.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)
P2_P3vsP60.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)

P11vsP2_P3.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)
P11vsP60.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)

P60vsP2_P3.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)
P60vsP11.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)

In [ ]:
conditions = {'P2_P3': [P2_P3vsP11, P2_P3vsP60], 'P11': [P11vsP2_P3, P11vsP60], 'P60':[P60vsP2_P3, P60vsP11]}

In [ ]:
genes = filter_de_genes(conditions, 'names', 4)

In [ ]:
sc.pl.heatmap(Endothelial,var_names=genes,groupby="Age", show_gene_labels=True,use_raw=False,
              layer='scaled',dendrogram=False , swap_axes=True, vmax=-4, vmin=4, cmap='BrBG_r',figsize=(4,15), 
              save='_Endothelial_no_prolif_downsampling.svg')

# Mesenchymal cells

In [ ]:
Mesenchymal = sc.read_h5ad('Mesenchymal_DE.h5ad')

In [ ]:
P2_P3vsP11 = pd.read_csv('Mesenchymal_P2_P3vsP11_markers.csv')
P2_P3vsP60 = pd.read_csv('Mesenchymal_P2_P3vsP60_markers.csv')

P11vsP2_P3 = pd.read_csv('Mesenchymal_P11vsP2_P3_markers.csv')
P11vsP60 = pd.read_csv('Mesenchymal_P11vsP60_markers.csv')

P60vsP2_P3 = pd.read_csv('Mesenchymal_P60vsP2_P3_markers.csv')
P60vsP11 = pd.read_csv('Mesenchymal_P60vsP11_markers.csv')

In [ ]:
P2_P3vsP11.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)
P2_P3vsP60.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)

P11vsP2_P3.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)
P11vsP60.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)

P60vsP2_P3.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)
P60vsP11.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)

In [ ]:
conditions = {'P2_P3': [P2_P3vsP11, P2_P3vsP60], 'P11': [P11vsP2_P3, P11vsP60], 'P60':[P60vsP2_P3, P60vsP11]}

In [ ]:
genes = filter_de_genes(conditions, 'names', 6)

In [ ]:
sc.pl.heatmap(Mesenchymal,var_names=genes,groupby="Age", show_gene_labels=True,use_raw=False,
              layer='scaled',dendrogram=False , swap_axes=True, vmax=-4, vmin=4, cmap='BrBG_r',figsize=(4,15), 
              save='_Mesenchymal_downsampling.svg')

# Fibroblasts

In [ ]:
Fibroblasts = sc.read_h5ad('Fibroblasts_DE.h5ad')

In [ ]:
P2_P3vsP11 = pd.read_csv('Fibroblasts_P2_P3vsP11_markers.csv')
P2_P3vsP60 = pd.read_csv('Fibroblasts_P2_P3vsP60_markers.csv')

P11vsP2_P3 = pd.read_csv('Fibroblasts_P11vsP2_P3_markers.csv')
P11vsP60 = pd.read_csv('Fibroblasts_P11vsP60_markers.csv')

P60vsP2_P3 = pd.read_csv('Fibroblasts_P60vsP2_P3_markers.csv')
P60vsP11 = pd.read_csv('Fibroblasts_P60vsP11_markers.csv')

In [ ]:
P2_P3vsP11.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)
P2_P3vsP60.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)

P11vsP2_P3.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)
P11vsP60.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)

P60vsP2_P3.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)
P60vsP11.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)

In [ ]:
conditions = {'P2_P3': [P2_P3vsP11, P2_P3vsP60], 'P11': [P11vsP2_P3, P11vsP60], 'P60':[P60vsP2_P3, P60vsP11]}

In [ ]:
genes = filter_de_genes(conditions, 'names', 6)

In [ ]:
sc.pl.heatmap(Fibroblasts,var_names=genes,groupby="Age", show_gene_labels=True,use_raw=False,
              layer='scaled',dendrogram=False , swap_axes=True, vmax=-4, vmin=4, cmap='BrBG_r',figsize=(4,15), 
              save='_Fibroblasts_downsampling.svg')

# Immune cells

In [ ]:
Immune_cells = sc.read_h5ad('Immune_cells_DE.h5ad')

In [ ]:
P2_P3vsP11 = pd.read_csv('Immune_cells_P2_P3vsP11_markers.csv')
P2_P3vsP60 = pd.read_csv('Immune_cells_P2_P3vsP60_markers.csv')

P11vsP2_P3 = pd.read_csv('Immune_cells_P11vsP2_P3_markers.csv')
P11vsP60 = pd.read_csv('Immune_cells_P11vsP60_markers.csv')

P60vsP2_P3 = pd.read_csv('Immune_cells_P60vsP2_P3_markers.csv')
P60vsP11 = pd.read_csv('Immune_cells_P60vsP11_markers.csv')

In [ ]:
P2_P3vsP11.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)
P2_P3vsP60.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)

P11vsP2_P3.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)
P11vsP60.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)

P60vsP2_P3.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)
P60vsP11.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)

In [ ]:
conditions = {'P2_P3': [P2_P3vsP11, P2_P3vsP60], 'P11': [P11vsP2_P3, P11vsP60], 'P60':[P60vsP2_P3, P60vsP11]}

In [ ]:
genes = filter_de_genes(conditions, 'names', 4)

In [ ]:
sc.pl.heatmap(Immune_cells,var_names=genes,groupby="Age", show_gene_labels=True,use_raw=False,
              layer='scaled',dendrogram=False , swap_axes=True, vmax=-4, vmin=4, cmap='BrBG_r',figsize=(4,15), 
              save='_Immune_cells_downsampling.svg')

# Macrophages

In [ ]:
Macrophages = sc.read_h5ad('Macrophages_DE.h5ad')

In [ ]:
P2_P3vsP11 = pd.read_csv('MacrophagesP2_P3vsP11_markers.csv')
P2_P3vsP60 = pd.read_csv('MacrophagesP2_P3vsP60_markers.csv')

P11vsP2_P3 = pd.read_csv('MacrophagesP11vsP2_P3_markers.csv')
P11vsP60 = pd.read_csv('MacrophagesP11vsP60_markers.csv')

P60vsP2_P3 = pd.read_csv('MacrophagesP60vsP2_P3_markers.csv')
P60vsP11 = pd.read_csv('MacrophagesP60vsP11_markers.csv')

In [ ]:
P2_P3vsP11.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)
P2_P3vsP60.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)

P11vsP2_P3.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)
P11vsP60.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)

P60vsP2_P3.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)
P60vsP11.drop(['Unnamed: 0', 'pts'], axis=1, inplace=True)

In [ ]:
conditions = {'P2_P3': [P2_P3vsP11, P2_P3vsP60], 'P11': [P11vsP2_P3, P11vsP60], 'P60':[P60vsP2_P3, P60vsP11]}

In [ ]:
genes = filter_de_genes(conditions, 'names', 4)

In [ ]:
sc.pl.heatmap(Macrophages,var_names=genes,groupby="Age", show_gene_labels=True,use_raw=False,
              layer='scaled',dendrogram=False , swap_axes=True, vmax=-4, vmin=4, cmap='BrBG_r',figsize=(4,15), 
              save='_Macrophages_downsampling.svg')